In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

# adjust this to match your actual Kaggle input path once attached
BASE_DIR = '/kaggle/input/datasets/shikhayadav78/raw-dataset'  # <-- update folder name if different

print("Top-level structure:")
for root, dirs, files in os.walk(BASE_DIR):
    depth = root.replace(BASE_DIR, '').count(os.sep)
    if depth <= 2:
        print(f"{'  ' * depth}{root} -> {len(files)} files, {len(dirs)} subdirs")
    if depth > 2:
        break

# count total images
total_images = 0
for root, dirs, files in os.walk(BASE_DIR):
    total_images += sum(1 for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png')))

print(f"\nTotal image files found: {total_images}")

# grab one sample path to check naming convention
for root, dirs, files in os.walk(BASE_DIR):
    imgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if imgs:
        print(f"\nSample folder: {root}")
        print("Sample filenames:", imgs[:5])
        break

In [ ]:
import os
import re
import pandas as pd

BASE_DIR = '/kaggle/input/datasets/shikhayadav78/raw-dataset/Categorized_AbbrvName'

records = []
# filename pattern: {diag_abbrv}_f{fitzpatrick}_{index}_{hash}.jpg
pattern = re.compile(r'^(.+)_f(\d)_(\d+)_([0-9a-f]+)\.jpg$', re.IGNORECASE)

for folder in os.listdir(BASE_DIR):
    folder_path = os.path.join(BASE_DIR, folder)
    if not os.path.isdir(folder_path):
        continue
    for fname in os.listdir(folder_path):
        if not fname.lower().endswith('.jpg'):
            continue
        m = pattern.match(fname)
        if m:
            diag_abbrv, fst, idx, h = m.groups()
            records.append({
                'diag_abbrv': diag_abbrv,
                'fitzpatrick_scale': int(fst),
                'filepath': os.path.join(folder_path, fname),
                'filename': fname,
                'folder': folder,
            })
        else:
            # flag anything that doesn't match expected pattern
            records.append({
                'diag_abbrv': None,
                'fitzpatrick_scale': None,
                'filepath': os.path.join(folder_path, fname),
                'filename': fname,
                'folder': folder,
            })

df = pd.DataFrame(records)
print(f"Total rows: {len(df)}")
print(f"Rows with unparseable filename: {df['fitzpatrick_scale'].isna().sum()}")

print("\nFitzpatrick scale distribution (parsed from filenames):")
print(df['fitzpatrick_scale'].value_counts().sort_index())

df.to_csv('raw_dataset_index.csv', index=False)
print("\nSaved raw_dataset_index.csv")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

df = pd.read_csv('raw_dataset_index.csv')

# drop unlabeled (fitzpatrick_scale == 0)
df = df[df['fitzpatrick_scale'] != 0].reset_index(drop=True)

def fst_band(x):
    if x in [1, 2]:
        return 'light'
    elif x in [3, 4]:
        return 'medium'
    else:
        return 'dark'

df['fst_band'] = df['fitzpatrick_scale'].apply(fst_band)

# visual grid
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
for row_idx, band in enumerate(['light', 'medium', 'dark']):
    subset = df[df['fst_band'] == band].sample(5, random_state=42)
    for col_idx, (_, row) in enumerate(subset.iterrows()):
        img = Image.open(row['filepath'])
        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].set_title(
            f"FST {row['fitzpatrick_scale']} ({band})\n{row['diag_abbrv'][:20]}",
            fontsize=9
        )
        axes[row_idx, col_idx].axis('off')
plt.tight_layout()
plt.savefig('sanity_check_raw_grid.png', dpi=100, bbox_inches='tight')
print("Saved sanity_check_raw_grid.png")

# quantitative check - same as before, but expect near-zero black/white fill this time
print("\n--- Quantitative check (raw images) ---")
sample_per_band = df.groupby('fst_band').apply(
    lambda x: x.sample(min(100, len(x)), random_state=42)
).reset_index(drop=True)

stats = []
for _, row in sample_per_band.iterrows():
    img = Image.open(row['filepath']).convert('RGB')
    arr = np.array(img)
    near_white = np.mean(np.all(arr > 240, axis=-1))
    near_black = np.mean(np.all(arr < 15, axis=-1))
    stats.append({
        'fst_band': row['fst_band'],
        'width': img.width,
        'height': img.height,
        'near_white_frac': near_white,
        'near_black_frac': near_black,
    })

stats_df = pd.DataFrame(stats)
print(stats_df.groupby('fst_band')[['width', 'height', 'near_white_frac', 'near_black_frac']].mean().round(3))

In [ ]:
import pandas as pd
import re

raw_df = pd.read_csv('raw_dataset_index.csv')
orig_df = pd.read_csv('/kaggle/input/datasets/nazmusresan/fitzpatrick17k/New folder/fitzpatrick17k (1).csv')

# extract the truncated 8-char hash from each filename
def extract_hash(fname):
    m = re.match(r'^.+_f\d_\d+_([0-9a-f]{8})\.jpg$', fname, re.IGNORECASE)
    return m.group(1) if m else None

raw_df['hash8'] = raw_df['filename'].apply(extract_hash)

# build a lookup: first 8 chars of md5hash -> full row
orig_df['hash8'] = orig_df['md5hash'].astype(str).str[:8]

# check for collisions on the 8-char prefix before merging
dupe_hashes = orig_df['hash8'].duplicated().sum()
print(f"Hash8 collisions in original CSV: {dupe_hashes}")

merged = raw_df.merge(
    orig_df[['hash8', 'label', 'three_partition_label', 'nine_partition_label', 'md5hash']],
    on='hash8', how='left'
)

print(f"\nTotal rows: {len(merged)}")
print(f"Rows successfully matched to original CSV: {merged['label'].notna().sum()}")
print(f"Rows with no match: {merged['label'].isna().sum()}")

print("\nThree-partition label distribution (after join):")
print(merged['three_partition_label'].value_counts())

merged.to_csv('fitzpatrick17k_raw_merged.csv', index=False)
print("\nSaved fitzpatrick17k_raw_merged.csv")
print("\nSample rows:")
print(merged[['filename', 'diag_abbrv', 'label', 'three_partition_label', 'fitzpatrick_scale']].head(10))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('fitzpatrick17k_raw_merged.csv')

# drop unlabeled skin tone rows (fitzpatrick_scale == 0, equivalent to -1 in original CSV)
df = df[df['fitzpatrick_scale'] != 0].reset_index(drop=True)
print(f"Usable rows after dropping unlabeled FST: {len(df)}")

def fst_band(x):
    if x in [1, 2]:
        return 'light'
    elif x in [3, 4]:
        return 'medium'
    else:
        return 'dark'

df['fst_band'] = df['fitzpatrick_scale'].apply(fst_band)
df['strat_key'] = df['three_partition_label'] + '_' + df['fst_band']

# check for any strat_key group too small to split (need at least 2 per group,
# and ideally more for a clean 70/15/15 three-way split)
group_sizes = df['strat_key'].value_counts()
print("\nSmallest strat groups:")
print(group_sizes.sort_values().head(10))

train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df['strat_key'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['strat_key'], random_state=42
)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

train_df.to_csv('train_split_raw.csv', index=False)
val_df.to_csv('val_split_raw.csv', index=False)
test_df.to_csv('test_split_raw.csv', index=False)
print("\nSaved train_split_raw.csv, val_split_raw.csv, test_split_raw.csv")

print("\n--- FST band balance across splits (should match closely) ---")
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"\n{name}:")
    print(d['fst_band'].value_counts(normalize=True).round(3))

print("\n--- Three-partition label balance across splits ---")
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"\n{name}:")
    print(d['three_partition_label'].value_counts(normalize=True).round(3))

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ---------- Label encoding ----------
LABEL_MAP = {'non-neoplastic': 0, 'benign': 1, 'malignant': 2}
LABEL_NAMES = ['non-neoplastic', 'benign', 'malignant']

# ---------- Dataset ----------
class SkinLesionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['filepath']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = LABEL_MAP[row['three_partition_label']]
        return img, label, row['fst_band']  # fst_band returned for later fairness eval

# ---------- Transforms ----------
# ImageNet normalization since we're using pretrained weights
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ---------- Load data ----------
train_df = pd.read_csv('train_split_raw.csv')
val_df = pd.read_csv('val_split_raw.csv')
test_df = pd.read_csv('test_split_raw.csv')

train_ds = SkinLesionDataset(train_df, transform=train_transform)
val_ds = SkinLesionDataset(val_df, transform=eval_transform)
test_ds = SkinLesionDataset(test_df, transform=eval_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ---------- Model ----------
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 3)  # 3 classes
model = model.to(device)

# ---------- Class-weighted loss (handles the 73/13.5/13.5 imbalance) ----------
class_counts = train_df['three_partition_label'].value_counts()
weights = torch.tensor([
    len(train_df) / (3 * class_counts['non-neoplastic']),
    len(train_df) / (3 * class_counts['benign']),
    len(train_df) / (3 * class_counts['malignant']),
], dtype=torch.float32).to(device)
print(f"Class weights: {weights}")

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# ---------- Training loop ----------
NUM_EPOCHS = 15
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    start = time.time()
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for imgs, labels, _ in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels, _ in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    train_loss /= train_total
    val_loss /= val_total
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    scheduler.step(val_loss)

    elapsed = time.time() - start
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} ({elapsed:.0f}s) | "
          f"Train loss: {train_loss:.4f} acc: {train_acc:.4f} | "
          f"Val loss: {val_loss:.4f} acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'baseline_efficientnet_b0.pt')
        print("  -> saved new best model")

print("\nTraining complete. Best model saved to baseline_efficientnet_b0.pt")

# ---------- Final test evaluation ----------
model.load_state_dict(torch.load('baseline_efficientnet_b0.pt'))
model.eval()

all_preds, all_labels, all_probs, all_fst_bands = [], [], [], []
with torch.no_grad():
    for imgs, labels, fst_bands in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_fst_bands.extend(fst_bands)

print("\n=== Overall Test Set Performance ===")
print(classification_report(all_labels, all_preds, target_names=LABEL_NAMES))
print(f"Overall accuracy: {accuracy_score(all_labels, all_preds):.4f}")

# save predictions for the fairness audit (next step)
results_df = test_df.copy().reset_index(drop=True)
results_df['pred'] = all_preds
results_df['pred_label'] = [LABEL_NAMES[p] for p in all_preds]
results_df['prob_malignant'] = [p[2] for p in all_probs]
results_df.to_csv('test_predictions_baseline.csv', index=False)
print("\nSaved test_predictions_baseline.csv for fairness audit step")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ---------- Label encoding ----------
LABEL_MAP = {'non-neoplastic': 0, 'benign': 1, 'malignant': 2}
LABEL_NAMES = ['non-neoplastic', 'benign', 'malignant']

# ---------- Dataset ----------
class SkinLesionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['filepath']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = LABEL_MAP[row['three_partition_label']]
        return img, label, row['fst_band']  # fst_band returned for later fairness eval

# ---------- Transforms ----------
# ImageNet normalization since we're using pretrained weights
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ---------- Load data ----------
train_df = pd.read_csv('train_split_raw.csv')
val_df = pd.read_csv('val_split_raw.csv')
test_df = pd.read_csv('test_split_raw.csv')

train_ds = SkinLesionDataset(train_df, transform=train_transform)
val_ds = SkinLesionDataset(val_df, transform=eval_transform)
test_ds = SkinLesionDataset(test_df, transform=eval_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ---------- Model ----------
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# freeze early feature layers - only fine-tune the later blocks + classifier.
# with only ~11k training images, fine-tuning the entire backbone from epoch 1
# overfits fast (this is what caused the train/val divergence above).
for name, param in model.features.named_parameters():
    # features has blocks 0-8; freeze blocks 0-5, fine-tune 6-8 + classifier
    block_num = name.split('.')[0]
    if block_num.isdigit() and int(block_num) < 6:
        param.requires_grad = False

num_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),  # increased from default 0.2 to fight overfitting
    nn.Linear(num_features, 3),
)
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

# ---------- Class-weighted loss (handles the 73/13.5/13.5 imbalance) ----------
class_counts = train_df['three_partition_label'].value_counts()
weights = torch.tensor([
    len(train_df) / (3 * class_counts['non-neoplastic']),
    len(train_df) / (3 * class_counts['benign']),
    len(train_df) / (3 * class_counts['malignant']),
], dtype=torch.float32).to(device)
print(f"Class weights: {weights}")

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-4  # L2 regularization to fight overfitting
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# ---------- Training loop ----------
NUM_EPOCHS = 15
best_val_loss = float('inf')
patience = 4
epochs_no_improve = 0

for epoch in range(NUM_EPOCHS):
    start = time.time()
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for imgs, labels, _ in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels, _ in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    train_loss /= train_total
    val_loss /= val_total
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    scheduler.step(val_loss)

    elapsed = time.time() - start
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} ({elapsed:.0f}s) | "
          f"Train loss: {train_loss:.4f} acc: {train_acc:.4f} | "
          f"Val loss: {val_loss:.4f} acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'baseline_efficientnet_b0.pt')
        print("  -> saved new best model")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs (no val improvement for {patience} epochs)")
            break

print("\nTraining complete. Best model saved to baseline_efficientnet_b0.pt")

# ---------- Final test evaluation ----------
model.load_state_dict(torch.load('baseline_efficientnet_b0.pt'))
model.eval()

all_preds, all_labels, all_probs, all_fst_bands = [], [], [], []
with torch.no_grad():
    for imgs, labels, fst_bands in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_fst_bands.extend(fst_bands)

print("\n=== Overall Test Set Performance ===")
print(classification_report(all_labels, all_preds, target_names=LABEL_NAMES))
print(f"Overall accuracy: {accuracy_score(all_labels, all_preds):.4f}")

# save predictions for the fairness audit (next step)
results_df = test_df.copy().reset_index(drop=True)
results_df['pred'] = all_preds
results_df['pred_label'] = [LABEL_NAMES[p] for p in all_preds]
results_df['prob_malignant'] = [p[2] for p in all_probs]
results_df.to_csv('test_predictions_baseline.csv', index=False)
print("\nSaved test_predictions_baseline.csv for fairness audit step")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ---------- Label encoding ----------
LABEL_MAP = {'non-neoplastic': 0, 'benign': 1, 'malignant': 2}
LABEL_NAMES = ['non-neoplastic', 'benign', 'malignant']

# ---------- Dataset ----------
class SkinLesionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['filepath']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = LABEL_MAP[row['three_partition_label']]
        return img, label, row['fst_band']  # fst_band returned for later fairness eval

# ---------- Transforms ----------
# ImageNet normalization since we're using pretrained weights
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ---------- Load data ----------
train_df = pd.read_csv('train_split_raw.csv')
val_df = pd.read_csv('val_split_raw.csv')
test_df = pd.read_csv('test_split_raw.csv')

train_ds = SkinLesionDataset(train_df, transform=train_transform)
val_ds = SkinLesionDataset(val_df, transform=eval_transform)
test_ds = SkinLesionDataset(test_df, transform=eval_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ---------- Model ----------
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# freeze early feature layers - only fine-tune the later blocks + classifier.
# with only ~11k training images, fine-tuning the entire backbone from epoch 1
# overfits fast (this is what caused the train/val divergence above).
for name, param in model.features.named_parameters():
    # features has blocks 0-8; freeze blocks 0-5, fine-tune 6-8 + classifier
    block_num = name.split('.')[0]
    if block_num.isdigit() and int(block_num) < 6:
        param.requires_grad = False

num_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),  # increased from default 0.2 to fight overfitting
    nn.Linear(num_features, 3),
)
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

# ---------- Class-weighted loss (handles the 73/13.5/13.5 imbalance) ----------
class_counts = train_df['three_partition_label'].value_counts()
weights = torch.tensor([
    len(train_df) / (3 * class_counts['non-neoplastic']),
    len(train_df) / (3 * class_counts['benign']),
    len(train_df) / (3 * class_counts['malignant']),
], dtype=torch.float32).to(device)
print(f"Class weights: {weights}")

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-4  # L2 regularization to fight overfitting
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# ---------- Training loop ----------
NUM_EPOCHS = 15
best_val_loss = float('inf')
patience = 4
epochs_no_improve = 0

for epoch in range(NUM_EPOCHS):
    start = time.time()
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for imgs, labels, _ in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels, _ in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    train_loss /= train_total
    val_loss /= val_total
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    scheduler.step(val_loss)

    elapsed = time.time() - start
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} ({elapsed:.0f}s) | "
          f"Train loss: {train_loss:.4f} acc: {train_acc:.4f} | "
          f"Val loss: {val_loss:.4f} acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'baseline_efficientnet_b0.pt')
        print("  -> saved new best model")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs (no val improvement for {patience} epochs)")
            break

print("\nTraining complete. Best model saved to baseline_efficientnet_b0.pt")

# ---------- Final test evaluation ----------
model.load_state_dict(torch.load('baseline_efficientnet_b0.pt'))
model.eval()

all_preds, all_labels, all_probs, all_fst_bands = [], [], [], []
with torch.no_grad():
    for imgs, labels, fst_bands in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_fst_bands.extend(fst_bands)

print("\n=== Overall Test Set Performance ===")
print(classification_report(all_labels, all_preds, target_names=LABEL_NAMES))
print(f"Overall accuracy: {accuracy_score(all_labels, all_preds):.4f}")

# save predictions for the fairness audit (next step)
results_df = test_df.copy().reset_index(drop=True)
results_df['pred'] = all_preds
results_df['pred_label'] = [LABEL_NAMES[p] for p in all_preds]
results_df['prob_malignant'] = [p[2] for p in all_probs]
results_df.to_csv('test_predictions_baseline.csv', index=False)
print("\nSaved test_predictions_baseline.csv for fairness audit step")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt

df = pd.read_csv('test_predictions_baseline.csv')
LABEL_MAP = {'non-neoplastic': 0, 'benign': 1, 'malignant': 2}
LABEL_NAMES = ['non-neoplastic', 'benign', 'malignant']
df['true'] = df['three_partition_label'].map(LABEL_MAP)

print(f"Total test samples: {len(df)}")
print(df['fst_band'].value_counts())

# ---------- 1. Overall accuracy by FST band ----------
print("\n" + "=" * 60)
print("OVERALL ACCURACY BY SKIN TONE BAND")
print("=" * 60)
for band in ['light', 'medium', 'dark']:
    sub = df[df['fst_band'] == band]
    acc = accuracy_score(sub['true'], sub['pred'])
    print(f"{band:8s} (n={len(sub):4d}): accuracy = {acc:.4f}")

# ---------- 2. Per-class metrics by FST band ----------
print("\n" + "=" * 60)
print("PER-CLASS PRECISION / RECALL / F1 BY SKIN TONE BAND")
print("=" * 60)
results = []
for band in ['light', 'medium', 'dark']:
    sub = df[df['fst_band'] == band]
    precision, recall, f1, support = precision_recall_fscore_support(
        sub['true'], sub['pred'], labels=[0, 1, 2], zero_division=0
    )
    for i, name in enumerate(LABEL_NAMES):
        results.append({
            'fst_band': band, 'class': name,
            'precision': precision[i], 'recall': recall[i],
            'f1': f1[i], 'support': support[i]
        })

results_df = pd.DataFrame(results)
pd.set_option('display.width', 120)
print(results_df.to_string(index=False))

# ---------- 3. THE KEY CLINICAL NUMBER: malignant recall by skin tone ----------
print("\n" + "=" * 60)
print("MALIGNANT RECALL BY SKIN TONE (most clinically important metric)")
print("=" * 60)
malignant_recall = results_df[results_df['class'] == 'malignant'][['fst_band', 'recall', 'support']]
print(malignant_recall.to_string(index=False))

# ---------- 4. Bootstrap confidence intervals ----------
# small sample sizes (esp. dark) mean point estimates alone can be misleading -
# bootstrap gives us a defensible range to report instead of a bare percentage
print("\n" + "=" * 60)
print("BOOTSTRAP 95% CI FOR MALIGNANT RECALL BY SKIN TONE (1000 resamples)")
print("=" * 60)

def bootstrap_recall_ci(sub_df, target_class=2, n_boot=1000, seed=42):
    rng = np.random.RandomState(seed)
    class_rows = sub_df[sub_df['true'] == target_class]
    if len(class_rows) == 0:
        return None, None, None
    recalls = []
    n = len(class_rows)
    for _ in range(n_boot):
        sample = class_rows.sample(n=n, replace=True, random_state=rng.randint(0, 1e6))
        recall = (sample['pred'] == target_class).mean()
        recalls.append(recall)
    return np.mean(recalls), np.percentile(recalls, 2.5), np.percentile(recalls, 97.5)

for band in ['light', 'medium', 'dark']:
    sub = df[df['fst_band'] == band]
    mean_r, lo, hi = bootstrap_recall_ci(sub)
    n_malignant = (sub['true'] == 2).sum()
    print(f"{band:8s} (n_malignant={n_malignant:3d}): recall = {mean_r:.3f}  95% CI [{lo:.3f}, {hi:.3f}]")

# ---------- 5. Confusion matrices per band ----------
print("\n" + "=" * 60)
print("CONFUSION MATRICES BY SKIN TONE BAND")
print("=" * 60)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, band in enumerate(['light', 'medium', 'dark']):
    sub = df[df['fst_band'] == band]
    cm = confusion_matrix(sub['true'], sub['pred'], labels=[0, 1, 2])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    axes[i].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    axes[i].set_title(f'{band} (n={len(sub)})')
    axes[i].set_xticks(range(3))
    axes[i].set_yticks(range(3))
    axes[i].set_xticklabels(LABEL_NAMES, rotation=45, ha='right')
    axes[i].set_yticklabels(LABEL_NAMES)
    for r in range(3):
        for c in range(3):
            axes[i].text(c, r, f'{cm_norm[r,c]:.2f}\n({cm[r,c]})',
                         ha='center', va='center', fontsize=8)
    if i == 0:
        axes[i].set_ylabel('True')
    axes[i].set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('fairness_confusion_matrices.png', dpi=100, bbox_inches='tight')
print("Saved fairness_confusion_matrices.png")

# ---------- 6. Save full results table ----------
results_df.to_csv('fairness_audit_results.csv', index=False)
print("\nSaved fairness_audit_results.csv")

# ---------- 7. Statistical significance test: is the light-vs-dark gap real? ----------
print("\n" + "=" * 60)
print("PERMUTATION TEST: is the light-vs-dark accuracy gap statistically significant?")
print("=" * 60)

def permutation_test(df, group_col, band_a, band_b, n_perm=5000, seed=42):
    rng = np.random.RandomState(seed)
    sub = df[df[group_col].isin([band_a, band_b])].copy()
    observed_diff = (
        accuracy_score(sub[sub[group_col] == band_a]['true'], sub[sub[group_col] == band_a]['pred']) -
        accuracy_score(sub[sub[group_col] == band_b]['true'], sub[sub[group_col] == band_b]['pred'])
    )
    labels = sub[group_col].values.copy()
    diffs = []
    for _ in range(n_perm):
        rng.shuffle(labels)
        mask_a = labels == band_a
        mask_b = labels == band_b
        acc_a = accuracy_score(sub['true'][mask_a], sub['pred'][mask_a])
        acc_b = accuracy_score(sub['true'][mask_b], sub['pred'][mask_b])
        diffs.append(acc_a - acc_b)
    diffs = np.array(diffs)
    p_value = np.mean(np.abs(diffs) >= np.abs(observed_diff))
    return observed_diff, p_value

obs_diff, p_val = permutation_test(df, 'fst_band', 'light', 'dark')
print(f"Light vs Dark accuracy gap: {obs_diff:.4f}")
print(f"Permutation test p-value: {p_val:.4f}")
print(f"{'SIGNIFICANT at p<0.05' if p_val < 0.05 else 'NOT statistically significant at p<0.05'}")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('test_predictions_baseline.csv')
LABEL_MAP = {'non-neoplastic': 0, 'benign': 1, 'malignant': 2}
df['true'] = df['three_partition_label'].map(LABEL_MAP)

def malignant_recall_permutation_test(df, band_a, band_b, n_perm=5000, seed=42):
    """
    Tests whether the malignant recall gap between two skin tone bands
    is larger than what we'd expect by chance, restricted to actual
    malignant cases only (recall = correctly caught / total actual malignant).
    """
    rng = np.random.RandomState(seed)
    # only look at true malignant cases from these two bands
    sub = df[(df['fst_band'].isin([band_a, band_b])) & (df['true'] == 2)].copy()

    def recall(sub_df, band):
        b = sub_df[sub_df['fst_band'] == band]
        if len(b) == 0:
            return np.nan
        return (b['pred'] == 2).mean()

    observed_diff = recall(sub, band_a) - recall(sub, band_b)

    labels = sub['fst_band'].values.copy()
    correct = (sub['pred'] == 2).values
    diffs = []
    for _ in range(n_perm):
        rng.shuffle(labels)
        mask_a = labels == band_a
        mask_b = labels == band_b
        r_a = correct[mask_a].mean()
        r_b = correct[mask_b].mean()
        diffs.append(r_a - r_b)
    diffs = np.array(diffs)
    p_value = np.mean(np.abs(diffs) >= np.abs(observed_diff))
    n_a = (sub['fst_band'] == band_a).sum()
    n_b = (sub['fst_band'] == band_b).sum()
    return observed_diff, p_value, n_a, n_b

print("=" * 65)
print("MALIGNANT RECALL GAP - PERMUTATION TESTS (pairwise)")
print("=" * 65)

for band_a, band_b in [('light', 'dark'), ('medium', 'dark'), ('light', 'medium')]:
    diff, p, n_a, n_b = malignant_recall_permutation_test(df, band_a, band_b)
    sig = "SIGNIFICANT (p<0.05)" if p < 0.05 else "not significant"
    print(f"\n{band_a} (n={n_a}) vs {band_b} (n={n_b}):")
    print(f"  Recall gap: {diff:+.3f}")
    print(f"  p-value: {p:.4f}  ->  {sig}")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

LABEL_MAP = {'non-neoplastic': 0, 'benign': 1, 'malignant': 2}
LABEL_NAMES = ['non-neoplastic', 'benign', 'malignant']

class SkinLesionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['filepath']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = LABEL_MAP[row['three_partition_label']]
        return img, label, row['fst_band']

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_df = pd.read_csv('train_split_raw.csv')
val_df = pd.read_csv('val_split_raw.csv')
test_df = pd.read_csv('test_split_raw.csv')

# ---------- Build per-sample weights for WeightedRandomSampler ----------
# weight each training sample by inverse frequency of its (label x fst_band)
# combination - this is the strat_key column already saved in the split.
# Rare groups (malignant_dark: 208 total, benign_dark: 203 total) get
# upsampled much more heavily than common groups (non-neoplastic_light: 5445).
group_counts = train_df['strat_key'].value_counts()
sample_weights = train_df['strat_key'].map(lambda k: 1.0 / group_counts[k]).values
sample_weights = torch.DoubleTensor(sample_weights)

print("\nSampling weight by group (higher = upsampled more):")
for k, c in group_counts.items():
    print(f"  {k:25s} count={c:5d}  weight={1.0/c:.6f}")

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),  # keep epoch size the same
    replacement=True
)

train_ds = SkinLesionDataset(train_df, transform=train_transform)
val_ds = SkinLesionDataset(val_df, transform=eval_transform)
test_ds = SkinLesionDataset(test_df, transform=eval_transform)

BATCH_SIZE = 32
# NOTE: sampler replaces shuffle=True - can't use both
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ---------- Model (same architecture/freezing as baseline) ----------
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
for name, param in model.features.named_parameters():
    block_num = name.split('.')[0]
    if block_num.isdigit() and int(block_num) < 6:
        param.requires_grad = False

num_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(num_features, 3),
)
model = model.to(device)

# NOTE: we keep class-weighted loss too - sampler balances skin tone x class
# jointly, but per-batch class loss weighting adds a second layer of signal.
# Using both together is standard practice (belt and suspenders), not redundant.
class_counts = train_df['three_partition_label'].value_counts()
weights = torch.tensor([
    len(train_df) / (3 * class_counts['non-neoplastic']),
    len(train_df) / (3 * class_counts['benign']),
    len(train_df) / (3 * class_counts['malignant']),
], dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

NUM_EPOCHS = 15
best_val_loss = float('inf')
patience = 4
epochs_no_improve = 0

for epoch in range(NUM_EPOCHS):
    start = time.time()
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for imgs, labels, _ in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels, _ in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    train_loss /= train_total
    val_loss /= val_total
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    scheduler.step(val_loss)

    elapsed = time.time() - start
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} ({elapsed:.0f}s) | "
          f"Train loss: {train_loss:.4f} acc: {train_acc:.4f} | "
          f"Val loss: {val_loss:.4f} acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'mitigated_efficientnet_b0.pt')
        print("  -> saved new best model")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break

print("\nTraining complete. Best model saved to mitigated_efficientnet_b0.pt")

# ---------- Test evaluation ----------
model.load_state_dict(torch.load('mitigated_efficientnet_b0.pt'))
model.eval()

all_preds, all_labels, all_fst_bands = [], [], []
with torch.no_grad():
    for imgs, labels, fst_bands in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_fst_bands.extend(fst_bands)

print("\n=== Overall Test Set Performance (Mitigated Model) ===")
print(classification_report(all_labels, all_preds, target_names=LABEL_NAMES))
print(f"Overall accuracy: {accuracy_score(all_labels, all_preds):.4f}")

results_df = test_df.copy().reset_index(drop=True)
results_df['pred'] = all_preds
results_df.to_csv('test_predictions_mitigated.csv', index=False)
print("\nSaved test_predictions_mitigated.csv")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt

df = pd.read_csv('test_predictions_mitigated.csv')
LABEL_MAP = {'non-neoplastic': 0, 'benign': 1, 'malignant': 2}
LABEL_NAMES = ['non-neoplastic', 'benign', 'malignant']
df['true'] = df['three_partition_label'].map(LABEL_MAP)

print(f"Total test samples: {len(df)}")
print(df['fst_band'].value_counts())

# ---------- 1. Overall accuracy by FST band ----------
print("\n" + "=" * 60)
print("OVERALL ACCURACY BY SKIN TONE BAND")
print("=" * 60)
for band in ['light', 'medium', 'dark']:
    sub = df[df['fst_band'] == band]
    acc = accuracy_score(sub['true'], sub['pred'])
    print(f"{band:8s} (n={len(sub):4d}): accuracy = {acc:.4f}")

# ---------- 2. Per-class metrics by FST band ----------
print("\n" + "=" * 60)
print("PER-CLASS PRECISION / RECALL / F1 BY SKIN TONE BAND")
print("=" * 60)
results = []
for band in ['light', 'medium', 'dark']:
    sub = df[df['fst_band'] == band]
    precision, recall, f1, support = precision_recall_fscore_support(
        sub['true'], sub['pred'], labels=[0, 1, 2], zero_division=0
    )
    for i, name in enumerate(LABEL_NAMES):
        results.append({
            'fst_band': band, 'class': name,
            'precision': precision[i], 'recall': recall[i],
            'f1': f1[i], 'support': support[i]
        })

results_df = pd.DataFrame(results)
pd.set_option('display.width', 120)
print(results_df.to_string(index=False))

# ---------- 3. THE KEY CLINICAL NUMBER: malignant recall by skin tone ----------
print("\n" + "=" * 60)
print("MALIGNANT RECALL BY SKIN TONE (most clinically important metric)")
print("=" * 60)
malignant_recall = results_df[results_df['class'] == 'malignant'][['fst_band', 'recall', 'support']]
print(malignant_recall.to_string(index=False))

# ---------- 4. Bootstrap confidence intervals ----------
# small sample sizes (esp. dark) mean point estimates alone can be misleading -
# bootstrap gives us a defensible range to report instead of a bare percentage
print("\n" + "=" * 60)
print("BOOTSTRAP 95% CI FOR MALIGNANT RECALL BY SKIN TONE (1000 resamples)")
print("=" * 60)

def bootstrap_recall_ci(sub_df, target_class=2, n_boot=1000, seed=42):
    rng = np.random.RandomState(seed)
    class_rows = sub_df[sub_df['true'] == target_class]
    if len(class_rows) == 0:
        return None, None, None
    recalls = []
    n = len(class_rows)
    for _ in range(n_boot):
        sample = class_rows.sample(n=n, replace=True, random_state=rng.randint(0, 1e6))
        recall = (sample['pred'] == target_class).mean()
        recalls.append(recall)
    return np.mean(recalls), np.percentile(recalls, 2.5), np.percentile(recalls, 97.5)

for band in ['light', 'medium', 'dark']:
    sub = df[df['fst_band'] == band]
    mean_r, lo, hi = bootstrap_recall_ci(sub)
    n_malignant = (sub['true'] == 2).sum()
    print(f"{band:8s} (n_malignant={n_malignant:3d}): recall = {mean_r:.3f}  95% CI [{lo:.3f}, {hi:.3f}]")

# ---------- 5. Confusion matrices per band ----------
print("\n" + "=" * 60)
print("CONFUSION MATRICES BY SKIN TONE BAND")
print("=" * 60)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, band in enumerate(['light', 'medium', 'dark']):
    sub = df[df['fst_band'] == band]
    cm = confusion_matrix(sub['true'], sub['pred'], labels=[0, 1, 2])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    axes[i].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    axes[i].set_title(f'{band} (n={len(sub)})')
    axes[i].set_xticks(range(3))
    axes[i].set_yticks(range(3))
    axes[i].set_xticklabels(LABEL_NAMES, rotation=45, ha='right')
    axes[i].set_yticklabels(LABEL_NAMES)
    for r in range(3):
        for c in range(3):
            axes[i].text(c, r, f'{cm_norm[r,c]:.2f}\n({cm[r,c]})',
                         ha='center', va='center', fontsize=8)
    if i == 0:
        axes[i].set_ylabel('True')
    axes[i].set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('fairness_confusion_matrices_mitigated.png', dpi=100, bbox_inches='tight')
print("Saved fairness_confusion_matrices_mitigated.png")

# ---------- 6. Save full results table ----------
results_df.to_csv('fairness_audit_results_mitigated.csv', index=False)
print("\nSaved fairness_audit_results_mitigated.csv")

# ---------- 7. Statistical significance test: is the light-vs-dark gap real? ----------
print("\n" + "=" * 60)
print("PERMUTATION TEST: is the light-vs-dark accuracy gap statistically significant?")
print("=" * 60)

def permutation_test(df, group_col, band_a, band_b, n_perm=5000, seed=42):
    rng = np.random.RandomState(seed)
    sub = df[df[group_col].isin([band_a, band_b])].copy()
    observed_diff = (
        accuracy_score(sub[sub[group_col] == band_a]['true'], sub[sub[group_col] == band_a]['pred']) -
        accuracy_score(sub[sub[group_col] == band_b]['true'], sub[sub[group_col] == band_b]['pred'])
    )
    labels = sub[group_col].values.copy()
    diffs = []
    for _ in range(n_perm):
        rng.shuffle(labels)
        mask_a = labels == band_a
        mask_b = labels == band_b
        acc_a = accuracy_score(sub['true'][mask_a], sub['pred'][mask_a])
        acc_b = accuracy_score(sub['true'][mask_b], sub['pred'][mask_b])
        diffs.append(acc_a - acc_b)
    diffs = np.array(diffs)
    p_value = np.mean(np.abs(diffs) >= np.abs(observed_diff))
    return observed_diff, p_value

obs_diff, p_val = permutation_test(df, 'fst_band', 'light', 'dark')
print(f"Light vs Dark accuracy gap: {obs_diff:.4f}")
print(f"Permutation test p-value: {p_val:.4f}")
print(f"{'SIGNIFICANT at p<0.05' if p_val < 0.05 else 'NOT statistically significant at p<0.05'}")

In [ ]:
import os
import pandas as pd

# update this to match your actual Kaggle input path once attached
BASE_DIR = '/kaggle/input/datasets/shikhayadav78/ddi-skinderna'  # <-- update folder name if different, check !ls /kaggle/input/

print("=" * 60)
print("FOLDER STRUCTURE")
print("=" * 60)
for root, dirs, files in os.walk(BASE_DIR):
    depth = root.replace(BASE_DIR, '').count(os.sep)
    if depth <= 2:
        print(f"{'  ' * depth}{root} -> {len(files)} files, {len(dirs)} subdirs")

# find any CSV/metadata files
print("\n" + "=" * 60)
print("METADATA FILES FOUND")
print("=" * 60)
csv_files = []
for root, dirs, files in os.walk(BASE_DIR):
    for f in files:
        if f.lower().endswith(('.csv', '.json', '.xlsx')):
            csv_files.append(os.path.join(root, f))
            print(os.path.join(root, f))

# inspect each metadata file found
for csv_path in csv_files:
    print(f"\n--- Inspecting {csv_path} ---")
    try:
        if csv_path.endswith('.csv'):
            meta_df = pd.read_csv(csv_path)
            print(f"Shape: {meta_df.shape}")
            print(f"Columns: {list(meta_df.columns)}")
            print(meta_df.head(10))
        elif csv_path.endswith('.json'):
            meta_df = pd.read_json(csv_path)
            print(f"Shape: {meta_df.shape}")
            print(f"Columns: {list(meta_df.columns)}")
            print(meta_df.head(10))
    except Exception as e:
        print(f"Could not parse: {e}")

# find image files and sample filenames
print("\n" + "=" * 60)
print("IMAGE FILES")
print("=" * 60)
image_files = []
for root, dirs, files in os.walk(BASE_DIR):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_files.append(os.path.join(root, f))

print(f"Total image files: {len(image_files)}")
print("Sample filenames:")
for f in image_files[:10]:
    print(" ", f)

In [ ]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

DDI_DIR = '/kaggle/input/datasets/shikhayadav78/ddi-skinderna'

ddi = pd.read_csv(os.path.join(DDI_DIR, 'ddi_metadata.csv'))
print(f"DDI rows: {len(ddi)}")

# map skin_tone (12/34/56) to the same fst_band categories used for Fitzpatrick17k
SKIN_TONE_MAP = {12: 'light', 34: 'medium', 56: 'dark'}
ddi['fst_band'] = ddi['skin_tone'].map(SKIN_TONE_MAP)

# map malignant bool to three_partition_label (DDI has no non-neoplastic class)
ddi['three_partition_label'] = ddi['malignant'].map({True: 'malignant', False: 'benign'})

ddi['filepath'] = ddi['DDI_file'].apply(lambda f: os.path.join(DDI_DIR, f))
ddi['strat_key'] = ddi['three_partition_label'] + '_' + ddi['fst_band']

print("\nDDI distribution by fst_band x label:")
print(ddi['strat_key'].value_counts())

# stratified split: most goes to train-supplement, a held-out slice for
# independent fairness testing (30% held out - small dataset, but this is
# specifically the diverse, expertly-curated set, so even a small held-out
# slice is valuable signal)
ddi_train, ddi_test = train_test_split(
    ddi, test_size=0.3, stratify=ddi['strat_key'], random_state=42
)

print(f"\nDDI train-supplement: {len(ddi_train)} | DDI held-out test: {len(ddi_test)}")
print("\nDDI train-supplement distribution:")
print(ddi_train['strat_key'].value_counts())
print("\nDDI held-out test distribution:")
print(ddi_test['strat_key'].value_counts())

# ---------- Merge DDI train-supplement into the existing Fitzpatrick17k train split ----------
fitz_train = pd.read_csv('train_split_raw.csv')

# keep only columns needed for training, matching schema across both sources
cols_needed = ['filepath', 'three_partition_label', 'fst_band', 'strat_key']

fitz_train_subset = fitz_train[cols_needed].copy()
fitz_train_subset['source'] = 'fitzpatrick17k'

ddi_train_subset = ddi_train[cols_needed].copy()
ddi_train_subset['source'] = 'ddi'

combined_train = pd.concat([fitz_train_subset, ddi_train_subset], ignore_index=True)
print(f"\nCombined train set: {len(combined_train)} "
      f"({len(fitz_train_subset)} Fitzpatrick17k + {len(ddi_train_subset)} DDI)")

print("\nCombined train strat_key distribution (compare dark_malignant/dark_benign to before):")
print(combined_train['strat_key'].value_counts())

combined_train.to_csv('train_split_with_ddi.csv', index=False)
ddi_test.to_csv('ddi_holdout_test.csv', index=False)
print("\nSaved train_split_with_ddi.csv and ddi_holdout_test.csv")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

LABEL_MAP = {'non-neoplastic': 0, 'benign': 1, 'malignant': 2}
LABEL_NAMES = ['non-neoplastic', 'benign', 'malignant']

class SkinLesionDataset(Dataset):
    def __init__(self, df, transform=None, has_nonneoplastic=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['filepath']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = LABEL_MAP[row['three_partition_label']]
        return img, label, row['fst_band']

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ---------- Load data ----------
train_df = pd.read_csv('train_split_with_ddi.csv')
val_df = pd.read_csv('val_split_raw.csv')            # unchanged Fitzpatrick val
test_df = pd.read_csv('test_split_raw.csv')           # original Fitzpatrick test
ddi_test_df = pd.read_csv('ddi_holdout_test.csv')     # independent DDI test

# ---------- Weighted sampler, recomputed on the new combined counts ----------
group_counts = train_df['strat_key'].value_counts()
sample_weights = train_df['strat_key'].map(lambda k: 1.0 / group_counts[k]).values
sample_weights = torch.DoubleTensor(sample_weights)
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_ds = SkinLesionDataset(train_df, transform=train_transform)
val_ds = SkinLesionDataset(val_df, transform=eval_transform)
test_ds = SkinLesionDataset(test_df, transform=eval_transform)
ddi_test_ds = SkinLesionDataset(ddi_test_df, transform=eval_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
ddi_test_loader = DataLoader(ddi_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ---------- Model: same architecture, but LOAD the mitigated checkpoint instead of ImageNet weights ----------
model = models.efficientnet_b0(weights=None)  # architecture only, weights loaded below
for name, param in model.features.named_parameters():
    block_num = name.split('.')[0]
    if block_num.isdigit() and int(block_num) < 6:
        param.requires_grad = False

num_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(num_features, 3),
)
model.load_state_dict(torch.load('mitigated_efficientnet_b0.pt'))
model = model.to(device)
print("Loaded mitigated_efficientnet_b0.pt as starting point for fine-tuning")

class_counts = train_df['three_partition_label'].value_counts()
weights = torch.tensor([
    len(train_df) / (3 * class_counts['non-neoplastic']),
    len(train_df) / (3 * class_counts['benign']),
    len(train_df) / (3 * class_counts['malignant']),
], dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
# lower LR since we're continuing training, not starting fresh
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

NUM_EPOCHS = 10  # fewer needed since we're fine-tuning from a trained checkpoint
best_val_loss = float('inf')
patience = 4
epochs_no_improve = 0

for epoch in range(NUM_EPOCHS):
    start = time.time()
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for imgs, labels, _ in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels, _ in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    train_loss /= train_total
    val_loss /= val_total
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    scheduler.step(val_loss)

    elapsed = time.time() - start
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} ({elapsed:.0f}s) | "
          f"Train loss: {train_loss:.4f} acc: {train_acc:.4f} | "
          f"Val loss: {val_loss:.4f} acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'final_efficientnet_b0_with_ddi.pt')
        print("  -> saved new best model")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break

print("\nFine-tuning complete. Best model saved to final_efficientnet_b0_with_ddi.pt")
model.load_state_dict(torch.load('final_efficientnet_b0_with_ddi.pt'))
model.eval()

# ---------- Evaluate on ORIGINAL Fitzpatrick test set ----------
def evaluate(loader, name):
    all_preds, all_labels, all_fst_bands = [], [], []
    with torch.no_grad():
        for imgs, labels, fst_bands in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            all_fst_bands.extend(fst_bands)
    print(f"\n=== {name} ===")
    print(classification_report(all_labels, all_preds, target_names=LABEL_NAMES, zero_division=0))
    print(f"Overall accuracy: {accuracy_score(all_labels, all_preds):.4f}")
    return all_preds, all_labels, all_fst_bands

preds1, labels1, fst1 = evaluate(test_loader, "Fitzpatrick17k Test Set (original)")
results_df = test_df.copy().reset_index(drop=True)
results_df['pred'] = preds1
results_df.to_csv('test_predictions_final.csv', index=False)

preds2, labels2, fst2 = evaluate(ddi_test_loader, "DDI Held-Out Test Set (independent)")
ddi_results_df = ddi_test_df.copy().reset_index(drop=True)
ddi_results_df['pred'] = preds2
ddi_results_df.to_csv('ddi_test_predictions_final.csv', index=False)

print("\nSaved test_predictions_final.csv and ddi_test_predictions_final.csv")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

LABEL_MAP = {'non-neoplastic': 0, 'benign': 1, 'malignant': 2}
LABEL_NAMES = ['non-neoplastic', 'benign', 'malignant']

def malignant_recall_by_band(df, name):
    df = df.copy()
    df['true'] = df['three_partition_label'].map(LABEL_MAP)
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    for band in ['light', 'medium', 'dark']:
        sub = df[df['fst_band'] == band]
        malig = sub[sub['true'] == 2]
        if len(malig) == 0:
            print(f"{band:8s}: no malignant cases")
            continue
        recall = (malig['pred'] == 2).mean()
        print(f"{band:8s} (n_malignant={len(malig):3d}): recall = {recall:.3f}")

# ---------- Fitzpatrick17k test set (final model) ----------
fitz_final = pd.read_csv('test_predictions_final.csv')
malignant_recall_by_band(fitz_final, "FITZPATRICK17K TEST SET — Final Model (post-DDI fine-tune)")

# ---------- DDI held-out test set (final model) ----------
ddi_final = pd.read_csv('ddi_test_predictions_final.csv')
malignant_recall_by_band(ddi_final, "DDI HELD-OUT TEST SET — Final Model (independent check)")

# ---------- Full three-way comparison table on Fitzpatrick test set ----------
print(f"\n{'='*60}\nFULL COMPARISON: Baseline -> Reweighted -> DDI Fine-tuned\n{'='*60}")
print("(all evaluated on the same Fitzpatrick17k test set for consistency)\n")

baseline = pd.read_csv('test_predictions_baseline.csv')
mitigated = pd.read_csv('test_predictions_mitigated.csv')
final = pd.read_csv('test_predictions_final.csv')

comparison = []
for name, d in [('baseline', baseline), ('reweighted', mitigated), ('ddi_finetuned', final)]:
    d = d.copy()
    d['true'] = d['three_partition_label'].map(LABEL_MAP)
    for band in ['light', 'medium', 'dark']:
        sub = d[d['fst_band'] == band]
        malig = sub[sub['true'] == 2]
        recall = (malig['pred'] == 2).mean() if len(malig) > 0 else np.nan
        comparison.append({'model': name, 'fst_band': band, 'malignant_recall': recall, 'n': len(malig)})

comp_df = pd.DataFrame(comparison)
pivot = comp_df.pivot(index='fst_band', columns='model', values='malignant_recall')
pivot = pivot[['baseline', 'reweighted', 'ddi_finetuned']]
pivot = pivot.reindex(['light', 'medium', 'dark'])
print(pivot.round(3).to_string())

pivot['dark_vs_medium_gap'] = pivot['ddi_finetuned']  # placeholder, real gap computed below
gap_baseline = pivot.loc['medium', 'baseline'] - pivot.loc['dark', 'baseline']
gap_reweighted = pivot.loc['medium', 'reweighted'] - pivot.loc['dark', 'reweighted']
gap_final = pivot.loc['medium', 'ddi_finetuned'] - pivot.loc['dark', 'ddi_finetuned']

print(f"\nMedium-vs-Dark malignant recall gap:")
print(f"  Baseline:      {gap_baseline:+.3f}")
print(f"  Reweighted:    {gap_reweighted:+.3f}")
print(f"  DDI fine-tuned: {gap_final:+.3f}")

comp_df.to_csv('full_mitigation_comparison.csv', index=False)
print("\nSaved full_mitigation_comparison.csv")